In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader

# 1. 공백이나 특수문자 에러를 방지하기 위해 절대 경로 또는 정확한 상대 경로 세팅
file_path = "data/공무원보수규정(대통령령)(제36501호)(20260707).pdf"

# 2. 파이썬이 이 파일을 인식하고 있는지 먼저 확인
if not os.path.exists(file_path):
    print(f"❌ [에러] 파일을 찾을 수 없습니다! 현재 경로: {os.getcwd()}")

else:
    print("p 파일이 존재합니다! 로드를 시작합니다.")
    
    # 3. 파일이 존재할 때만 로더 실행
    loader = PyPDFLoader(file_path)
    docs = loader.load_and_split()

C:\Users\박동관\AppData\Local\Temp\ipykernel_54232\17869605.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


p 파일이 존재합니다! 로드를 시작합니다.


In [2]:
import re
from langchain_core.documents import Document

# 1. 이전 셀에서 로드한 모든 페이지의 텍스트를 하나로 묶기
full_text = "\n".join([doc.page_content for doc in docs])

# 2. 정규표현식을 이용해 "제 1 조" 또는 "제1조" 패턴 찾기
# [로직]: '제'로 시작하고 뒤에 숫자(들)가 오고 '조'로 끝나는 문장을 기준으로 자릅니다.
article_pattern = r"(제\s*\d+\s*조\([^\)]+\))|(제\s*\d+\s*조)"
articles = re.split(article_pattern, full_text)

# 3. 쪼개진 텍스트들을 정제하여 LangChain Document 객체로 재조립
chunked_docs = []
current_article_title = "제1조(목적)" # 기본값 세팅

# re.split을 쓰면 패턴과 내용이 번갈아가며 리스트에 들어갑니다.
for item in articles:
    if not item or item.strip() == "":
        continue
        
    # 만약 "제X조(제목)" 형태의 패턴을 발견했다면 타이틀로 보관
    if re.match(article_pattern, item.strip()):
        current_article_title = item.strip()
    else:
        # 패턴 뒤에 나오는 실제 법률 본문 내용인 경우
        content = item.strip()
        
        # 앞부분에 남아있을 수 있는 불필요한 줄바꿈이나 공백 정제
        if len(content) > 10: # 의미 있는 길이의 본문만 저장
            # 새롭게 조 단위로 매핑된 고품질 Document 객체 생성
            new_doc = Document(
                page_content=f"{current_article_title} {content}",
                metadata={
                    "user_type": "officer",
                    "law_name": "공무원보수규정",
                    "section": current_article_title # 메타데이터에 "제2조(정의)" 형태로 쏙 들어감
                }
            )
            chunked_docs.append(new_doc)

# 4. 결과 검증 (잘 쪼개졌는지 확인)
print(f"✂️ 조 단위로 최종 분할된 덩어리(Chunk) 개수: {len(chunked_docs)}개\n")

print("=== [검증] 첫 번째 조항 데이터 ===")
print(f"🔹 메타데이터: {chunked_docs[0].metadata}")
print(f"🔹 본문:\n{chunked_docs[0].page_content}\n")

print("=== [검증] 두 번째 조항 데이터 ===")
print(f"🔹 메타데이터: {chunked_docs[1].metadata}")
print(f"🔹 본문:\n{chunked_docs[1].page_content}")

✂️ 조 단위로 최종 분할된 덩어리(Chunk) 개수: 252개

=== [검증] 첫 번째 조항 데이터 ===
🔹 메타데이터: {'user_type': 'officer', 'law_name': '공무원보수규정', 'section': '제1조(목적)'}
🔹 본문:
제1조(목적) 법제처                                                            1                                                       국가법령정보센터
공무원보수규정
 
공무원보수규정
[시행 2026. 7. 7.] [대통령령 제36501호, 2026. 7. 7., 타법개정]
인사혁신처 (급여정책과) 044-201-8396
       제1장 총칙 <개정 2009. 3. 31.>

=== [검증] 두 번째 조항 데이터 ===
🔹 메타데이터: {'user_type': 'officer', 'law_name': '공무원보수규정', 'section': '제1조(목적)'}
🔹 본문:
제1조(목적) 이 영은 「국가공무원법」, 「헌법재판소법」, 「외무공무원법」, 「경찰공무원법」, 「의무경찰대 설치 및 운영에
관한 법률」, 「소방공무원법」, 「의무소방대설치법」, 「교육공무원법」, 「군인보수법」, 「군무원인사법」, 「국가정보원직
원법」 및 「군법무관 임용 등에 관한 법률」에 따라 국가공무원의 보수에 관한 사항을 규정함을 목적으로 한다. <개정
2013. 1. 9., 2015. 11. 20., 2016. 11. 29.>
[전문개정 2009. 3. 31.]


In [3]:
len(chunked_docs)

252